# ETF volatility baselines

Quick notebook to add 4 baseline columns by ticker: avg-5, avg-20, ols-5, ols-20.
Uses only values that would have been known at that date.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/vol_dataset_post_wrangle_021026.csv'
OUT_PATH  = '/content/drive/MyDrive/Colab Notebooks/vol_dataset_with_4_online_baselines.csv'

DATE_COL   = 'date'
TICKER_COL = 'ticker'
Y_COL      = 'forward_vol_5d_annual_decimel_calculated'
HORIZON    = 5  # forward window in trading days

In [ ]:
df0 = pd.read_csv(DATA_PATH, low_memory=False)

missing = {DATE_COL, TICKER_COL, Y_COL} - set(df0.columns)
assert not missing, f'missing cols: {missing}'

df = df0.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)

print(df.shape, '|', df[TICKER_COL].nunique(), 'tickers')
print(df[DATE_COL].min().date(), '->', df[DATE_COL].max().date())

## Known target at time t

`y(t)` looks forward 5 trading days, so I shift it by 5 rows within each ticker before building the baselines.

In [ ]:
df['y_known_at_t'] = df.groupby(TICKER_COL, sort=False)[Y_COL].shift(HORIZON)

# quick spot check
tkr = df[TICKER_COL].iloc[0]
df[df[TICKER_COL] == tkr][[DATE_COL, Y_COL, 'y_known_at_t']].head(10)


In [ ]:
g = df.groupby(TICKER_COL, sort=False)['y_known_at_t']

df['baseline_avg_5'] = g.rolling(5, min_periods=5).mean().reset_index(level=0, drop=True)
df['baseline_avg_20'] = g.rolling(20, min_periods=20).mean().reset_index(level=0, drop=True)

df[[DATE_COL, TICKER_COL, 'baseline_avg_5', 'baseline_avg_20']].head(10)


In [ ]:
def rolling_ols_forecast(series, window, horizon):
    x = np.arange(window, dtype=float)
    xm = x.mean()
    ss = ((x - xm) ** 2).sum()
    x_pred = (window - 1) + horizon

    def predict(y_vals):
        y = y_vals.astype(float)
        slope = ((x - xm) * (y - y.mean())).sum() / ss
        return y.mean() + slope * (x_pred - xm)

    return series.rolling(window, min_periods=window).apply(predict, raw=True)


g = df.groupby(TICKER_COL, sort=False)['y_known_at_t']

df['baseline_ols_5'] = (
    g.apply(lambda s: rolling_ols_forecast(s, 5, HORIZON))
     .reset_index(level=0, drop=True)
)

df['baseline_ols_20'] = (
    g.apply(lambda s: rolling_ols_forecast(s, 20, HORIZON))
     .reset_index(level=0, drop=True)
)

df[[DATE_COL, TICKER_COL, 'baseline_ols_5', 'baseline_ols_20']].head(10)


In [ ]:
bcols = ['baseline_avg_5', 'baseline_avg_20', 'baseline_ols_5', 'baseline_ols_20']

print('nan rates:')
print(df[bcols].isna().mean().round(3))

print('\nnegative OLS values:', (df['baseline_ols_5'] < 0).sum(), (df['baseline_ols_20'] < 0).sum())

tkr = df[TICKER_COL].iloc[0]
df[df[TICKER_COL] == tkr][[DATE_COL, Y_COL, 'y_known_at_t'] + bcols].head(30)


In [ ]:
df.to_csv(OUT_PATH, index=False)
print('saved:', OUT_PATH)